<a href="https://colab.research.google.com/github/Iamjuhwan/Deep-Learing/blob/main/Building_backpropagation_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ── Tiny 2-layer network ──────────────────────────────────
# XOR dataset

In [ ]:
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
y = np.array([[0],[1],[1],[0]],         dtype=np.float64)

In [ ]:
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5   # (2→4)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5   # (4→1)
b2 = np.zeros((1, 1))

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X):
    Z1 = X @ W1 + b1        # (4,4)
    A1 = sigmoid(Z1)        # (4,4)
    Z2 = A1 @ W2 + b2       # (4,1)
    A2 = sigmoid(Z2)        # (4,1)
    return Z1, A1, Z2, A2

In [ ]:
def backward(X, y, Z1, A1, Z2, A2):
    m = X.shape[0]
    # ── output layer ──────────────────────────────────────
    dZ2 = A2 - y                           # dL/dZ2  (4,1)
    dW2 = A1.T @ dZ2 / m                   # (4,1)
    db2 = dZ2.mean(axis=0, keepdims=True)  # (1,1)
    # ── hidden layer ──────────────────────────────────────
    dA1 = dZ2 @ W2.T                       # (4,4)
    dZ1 = dA1 * A1 * (1 - A1)             # sigmoid' = A1*(1-A1)
    dW1 = X.T @ dZ1 / m                   # (2,4)
    db1 = dZ1.mean(axis=0, keepdims=True)  # (1,4)
    return dW1, db1, dW2, db2

lr = 0.5
losses = []


In [ ]:
for epoch in range(1000):
    Z1, A1, Z2, A2 = forward(X)
    loss = -np.mean(y*np.log(A2+1e-9) + (1-y)*np.log(1-A2+1e-9))
    losses.append(loss)

    dW1, db1, dW2, db2 = backward(X, y, Z1, A1, Z2, A2)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

In [ ]:
# Final predictions
_, _, _, preds = forward(X)
print("Epoch 1000  loss:", round(float(losses[-1]), 6))
print()
print("Input → Predicted  (target)")
for i in range(4):
    print(f"  {X[i]} → {preds[i][0]:.4f}   (y={int(y[i][0])})")

Epoch 1000  loss: 0.594623

Input → Predicted  (target)
  [0. 0.] → 0.3205   (y=0)
  [0. 1.] → 0.5693   (y=1)
  [1. 0.] → 0.5608   (y=1)
  [1. 1.] → 0.5714   (y=0)


In [ ]:
# Export loss curve for the Visualize tab
import json, sys
print("__LOSSES__:" + json.dumps([round(float(l),6) for l in losses[::10]]))

__LOSSES__:[0.704555, 0.693856, 0.693782, 0.693726, 0.693674, 0.693626, 0.693582, 0.693542, 0.693504, 0.693469, 0.693436, 0.693405, 0.693376, 0.693349, 0.693324, 0.693299, 0.693276, 0.693254, 0.693232, 0.693212, 0.693192, 0.693173, 0.693154, 0.693136, 0.693118, 0.6931, 0.693083, 0.693065, 0.693048, 0.69303, 0.693012, 0.692994, 0.692976, 0.692958, 0.692939, 0.692919, 0.692899, 0.692878, 0.692856, 0.692833, 0.692809, 0.692784, 0.692758, 0.69273, 0.6927, 0.692668, 0.692634, 0.692598, 0.692559, 0.692517, 0.692471, 0.692422, 0.692368, 0.69231, 0.692246, 0.692176, 0.692099, 0.692014, 0.69192, 0.691816, 0.6917, 0.691571, 0.691427, 0.691265, 0.691084, 0.690879, 0.690649, 0.690388, 0.690092, 0.689757, 0.689375, 0.688941, 0.688447, 0.687884, 0.687242, 0.686512, 0.68568, 0.684736, 0.683664, 0.68245, 0.681078, 0.679532, 0.677795, 0.675847, 0.673669, 0.671242, 0.668544, 0.665553, 0.662248, 0.658607, 0.654608, 0.650235, 0.645473, 0.640313, 0.634753, 0.628797, 0.622461, 0.615766, 0.608743, 0.601426]


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.datasets import make_classification

In [ ]:
np.random.seed(0)

## 1. LINEAR REGRESSION FROM SCRATCH

In [ ]:
class LinearRegressionScratch:
    def __init__(self, lr=0.01, epochs=200, gd='batch', batch_size=16):
        self.lr = lr
        self.epochs = epochs
        self.gd = gd
        self.bs = batch_size
        self.losses = []

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0

        for _ in range(self.epochs):
            if self.gd == 'batch':
                idxs = np.arange(m)
            elif self.gd == 'sgd':
                idxs = np.random.choice(m, 1)
            else:  # mini-batch
                idxs = np.random.choice(m, self.bs, replace=False)

            Xb, yb = X[idxs], y[idxs]
            pred = Xb @ self.w + self.b
            err  = pred - yb

            dw = Xb.T @ err / len(idxs)
            db = err.mean()

            self.w -= self.lr * dw
            self.b -= self.lr * db

            full_pred = X @ self.w + self.b
            self.losses.append(np.mean((full_pred - y) ** 2))

        return self

    def predict(self, X):
        return X @ self.w + self.b

In [ ]:
X_lin = np.random.randn(100, 1)
y_lin = 3 * X_lin.ravel() + 2 + np.random.randn(100) * 0.5

scratch_lin = LinearRegressionScratch(lr=0.1, epochs=200, gd='batch').fit(X_lin, y_lin)
sklearn_lin = LinearRegression().fit(X_lin, y_lin)

y_pred_scratch = scratch_lin.predict(X_lin)
y_pred_sklearn = sklearn_lin.predict(X_lin)

print("=== Linear Regression ===")
print(f"  Scratch  MSE={mean_squared_error(y_lin, y_pred_scratch):.4f}  R²={r2_score(y_lin, y_pred_scratch):.4f}")
print(f"  sklearn  MSE={mean_squared_error(y_lin, y_pred_sklearn):.4f}  R²={r2_score(y_lin, y_pred_sklearn):.4f}")
print(f"  Scratch  w={scratch_lin.w[0]:.3f}  b={scratch_lin.b:.3f}")
print(f"  sklearn  w={sklearn_lin.coef_[0]:.3f}  b={sklearn_lin.intercept_:.3f}")

=== Linear Regression ===
  Scratch  MSE=0.2125  R²=0.9756
  sklearn  MSE=0.2125  R²=0.9756
  Scratch  w=3.062  b=1.905
  sklearn  w=3.062  b=1.905


In [ ]:
print("\n=== GD Variant Comparison (Linear, 200 epochs) ===")
for variant in ['batch', 'sgd', 'mini']:
    m = LinearRegressionScratch(lr=0.1, epochs=200, gd=variant).fit(X_lin, y_lin)
    final_mse = m.losses[-1]
    print(f"  {variant:<8}  final MSE={final_mse:.4f}")


=== GD Variant Comparison (Linear, 200 epochs) ===
  batch     final MSE=0.2125
  sgd       final MSE=0.2172
  mini      final MSE=0.2126


## 2. LOGISTIC REGRESSION FROM SCRATCH

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=200, gd='batch', batch_size=16):
        self.lr = lr
        self.epochs = epochs
        self.gd = gd
        self.bs = batch_size
        self.losses = []

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0

        for _ in range(self.epochs):
            if self.gd == 'batch':
                idxs = np.arange(m)
            elif self.gd == 'sgd':
                idxs = np.random.choice(m, 1)
            else:  # mini-batch
                idxs = np.random.choice(m, self.bs, replace=False)

            Xb, yb = X[idxs], y[idxs]
            p   = self.sigmoid(Xb @ self.w + self.b)
            err = p - yb

            dw = Xb.T @ err / len(idxs)
            db = err.mean()

            self.w -= self.lr * dw
            self.b -= self.lr * db

            full_p = self.sigmoid(X @ self.w + self.b)
            eps = 1e-9
            bce = -np.mean(y * np.log(full_p + eps) + (1 - y) * np.log(1 - full_p + eps))
            self.losses.append(bce)

        return self

    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
X_log, y_log = make_classification(
    n_samples=150, n_features=2, n_redundant=0,
    n_clusters_per_class=1, random_state=0
)

scratch_log = LogisticRegressionScratch(lr=0.1, epochs=200, gd='batch').fit(X_log, y_log)
sklearn_log = LogisticRegression(max_iter=1000).fit(X_log, y_log)

print("\n=== Logistic Regression ===")
print(f"  Scratch  acc={accuracy_score(y_log, scratch_log.predict(X_log)):.4f}")
print(f"  sklearn  acc={accuracy_score(y_log, sklearn_log.predict(X_log)):.4f}")
print(f"  Final BCE loss={scratch_log.losses[-1]:.4f}")


=== Logistic Regression ===
  Scratch  acc=0.8867
  sklearn  acc=0.9000
  Final BCE loss=0.2932


In [ ]:
print("\n=== GD Variant Comparison (Logistic, 200 epochs) ===")
for variant in ['batch', 'sgd', 'mini']:
    m = LogisticRegressionScratch(lr=0.1, epochs=200, gd=variant).fit(X_lin, y_lin)
    final_mse = m.losses[-1]
    print(f"  {variant:<8}  final MSE={final_mse:.4f}")


=== GD Variant Comparison (Logistic, 200 epochs) ===
  batch     final MSE=-41.7903
  sgd       final MSE=-41.8642
  mini      final MSE=-41.7484


## Building a 2-layer MLP on MNIST

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

# ══════════════════════════════════════════════════
#  1. LOAD & PREPROCESS MNIST
# ══════════════════════════════════════════════════

print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target.astype(int)

# Normalize to [0, 1]
X = X / 255.0

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One-hot encode labels  →  shape (m, 10)
enc = OneHotEncoder(sparse_output=False)
Y_train = enc.fit_transform(y_train.reshape(-1, 1))  # (56000, 10)
Y_test  = enc.transform(y_test.reshape(-1, 1))       # (14000, 10)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


# ══════════════════════════════════════════════════
#  2. ACTIVATIONS & THEIR DERIVATIVES
# ══════════════════════════════════════════════════

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(z):
    # Subtract row-max for numerical stability
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(probs, Y_onehot):
    m = Y_onehot.shape[0]
    log_p = np.log(probs + 1e-9)
    return -np.sum(Y_onehot * log_p) / m


# ══════════════════════════════════════════════════
#  3. WEIGHT INITIALIZATION
# ══════════════════════════════════════════════════

def init_weights(layer_dims, seed=42):
    """
    layer_dims: [input_size, hidden1, hidden2, ..., output_size]
    Uses He initialization for ReLU layers.
    """
    np.random.seed(seed)
    params = {}
    L = len(layer_dims) - 1  # number of layers

    for l in range(1, L + 1):
        fan_in = layer_dims[l - 1]
        # He init: scale by sqrt(2 / fan_in)  — optimal for ReLU
        params[f'W{l}'] = np.random.randn(layer_dims[l - 1], layer_dims[l]) * np.sqrt(2.0 / fan_in)
        params[f'b{l}'] = np.zeros((1, layer_dims[l]))

    return params


# ══════════════════════════════════════════════════
#  4. FORWARD PASS
# ══════════════════════════════════════════════════

def forward(X, params):
    """
    2-layer MLP:
      Layer 1: Linear → ReLU
      Layer 2: Linear → Softmax
    Returns predictions and cached values needed for backprop.
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']

    # Hidden layer
    Z1 = X @ W1 + b1          # (m, hidden)
    A1 = relu(Z1)              # (m, hidden)

    # Output layer
    Z2 = A1 @ W2 + b2         # (m, 10)
    A2 = softmax(Z2)           # (m, 10)  — probabilities

    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2, 'X': X}
    return A2, cache


# ══════════════════════════════════════════════════
#  5. BACKWARD PASS
# ══════════════════════════════════════════════════

def backward(params, cache, Y_onehot):
    """
    Computes gradients via backpropagation.
    Softmax + cross-entropy gradient simplifies to: dZ2 = A2 - Y
    """
    m  = Y_onehot.shape[0]
    W2 = params['W2']

    X  = cache['X']
    Z1 = cache['Z1']
    A1 = cache['A1']
    A2 = cache['A2']

    # ── Output layer gradients ────────────────────
    dZ2 = A2 - Y_onehot                        # (m, 10)
    dW2 = A1.T @ dZ2 / m                       # (hidden, 10)
    db2 = dZ2.mean(axis=0, keepdims=True)      # (1, 10)

    # ── Hidden layer gradients ────────────────────
    dA1 = dZ2 @ W2.T                           # (m, hidden)
    dZ1 = dA1 * relu_deriv(Z1)                 # (m, hidden)  — gate by ReLU'
    dW1 = X.T @ dZ1 / m                        # (784, hidden)
    db1 = dZ1.mean(axis=0, keepdims=True)      # (1, hidden)

    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads


# ══════════════════════════════════════════════════
#  6. GRADIENT DESCENT VARIANTS
# ══════════════════════════════════════════════════

def update_params(params, grads, lr):
    """Vanilla SGD / mini-batch GD update."""
    params['W1'] -= lr * grads['dW1']
    params['b1'] -= lr * grads['db1']
    params['W2'] -= lr * grads['dW2']
    params['b2'] -= lr * grads['db2']
    return params


def update_params_momentum(params, grads, velocity, lr, beta=0.9):
    """SGD + Momentum: smooths noisy gradients."""
    for key in ['W1', 'b1', 'W2', 'b2']:
        velocity[key] = beta * velocity[key] - lr * grads[f'd{key}']
        params[key]  += velocity[key]
    return params, velocity


def update_params_adam(params, grads, adam_state, lr, t,
                       beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam: adaptive learning rates per parameter."""
    for key in ['W1', 'b1', 'W2', 'b2']:
        g = grads[f'd{key}']
        adam_state['m'][key] = beta1 * adam_state['m'][key] + (1 - beta1) * g
        adam_state['v'][key] = beta2 * adam_state['v'][key] + (1 - beta2) * g**2
        # Bias correction
        m_hat = adam_state['m'][key] / (1 - beta1**t)
        v_hat = adam_state['v'][key] / (1 - beta2**t)
        params[key] -= lr * m_hat / (np.sqrt(v_hat) + eps)
    return params, adam_state


# ══════════════════════════════════════════════════
#  7. TRAINING LOOP
# ══════════════════════════════════════════════════

def train(X_train, Y_train, X_val, Y_val,
          layer_dims, lr=0.001, epochs=20,
          batch_size=128, optimizer='adam', seed=42):

    params   = init_weights(layer_dims, seed)
    history  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    m        = X_train.shape[0]

    # Optimizer state
    velocity   = {k: np.zeros_like(v) for k, v in params.items()}
    adam_state = {
        'm': {k: np.zeros_like(v) for k, v in params.items()},
        'v': {k: np.zeros_like(v) for k, v in params.items()}
    }
    t = 0  # Adam step counter

    for epoch in range(1, epochs + 1):
        # Shuffle training data each epoch
        perm    = np.random.permutation(m)
        X_shuf  = X_train[perm]
        Y_shuf  = Y_train[perm]

        # Mini-batch loop
        for start in range(0, m, batch_size):
            Xb = X_shuf[start:start + batch_size]
            Yb = Y_shuf[start:start + batch_size]

            probs, cache = forward(Xb, params)
            grads        = backward(params, cache, Yb)

            t += 1
            if optimizer == 'adam':
                params, adam_state = update_params_adam(params, grads, adam_state, lr, t)
            elif optimizer == 'momentum':
                params, velocity   = update_params_momentum(params, grads, velocity, lr)
            else:
                params             = update_params(params, grads, lr)

        # ── Epoch metrics ─────────────────────────
        train_probs, _ = forward(X_train, params)
        val_probs,   _ = forward(X_val,   params)

        train_loss = cross_entropy(train_probs, Y_train)
        val_loss   = cross_entropy(val_probs,   Y_val)
        train_acc  = (train_probs.argmax(axis=1) == Y_train.argmax(axis=1)).mean()
        val_acc    = (val_probs.argmax(axis=1)   == Y_val.argmax(axis=1)).mean()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:02d}/{epochs}  "
              f"loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"acc={train_acc:.4f}  val_acc={val_acc:.4f}")

    return params, history


# ══════════════════════════════════════════════════
#  8. PREDICT & EVALUATE
# ══════════════════════════════════════════════════

def predict(X, params):
    probs, _ = forward(X, params)
    return probs.argmax(axis=1)


# ══════════════════════════════════════════════════
#  9. RUN
# ══════════════════════════════════════════════════

layer_dims = [784, 256, 10]
#              ↑    ↑    ↑
#           input  h1  output (digits 0-9)

params, history = train(
    X_train, Y_train,
    X_test,  Y_test,
    layer_dims = layer_dims,
    lr         = 0.001,
    epochs     = 20,
    batch_size = 128,
    optimizer  = 'adam',   # 'adam' | 'momentum' | 'sgd'
)

# Final evaluation
y_pred = predict(X_test, params)
print("\n=== Test Set Report ===")
print(classification_report(y_test, y_pred))

# Final accuracy
final_acc = (y_pred == y_test).mean()
print(f"Final test accuracy: {final_acc*100:.2f}%")

Loading MNIST...
Train: (56000, 784), Test: (14000, 784)
Epoch 01/20  loss=0.1605  val_loss=0.1828  acc=0.9542  val_acc=0.9471
Epoch 02/20  loss=0.0939  val_loss=0.1211  acc=0.9738  val_acc=0.9646
Epoch 03/20  loss=0.0696  val_loss=0.1001  acc=0.9811  val_acc=0.9703
Epoch 04/20  loss=0.0505  val_loss=0.0853  acc=0.9861  val_acc=0.9739
Epoch 05/20  loss=0.0369  val_loss=0.0779  acc=0.9904  val_acc=0.9764
Epoch 06/20  loss=0.0300  val_loss=0.0750  acc=0.9924  val_acc=0.9775
Epoch 07/20  loss=0.0211  val_loss=0.0678  acc=0.9953  val_acc=0.9793
Epoch 08/20  loss=0.0206  val_loss=0.0742  acc=0.9956  val_acc=0.9784
Epoch 09/20  loss=0.0149  val_loss=0.0700  acc=0.9971  val_acc=0.9793
Epoch 10/20  loss=0.0119  val_loss=0.0734  acc=0.9978  val_acc=0.9781
Epoch 11/20  loss=0.0103  val_loss=0.0699  acc=0.9982  val_acc=0.9799
Epoch 12/20  loss=0.0086  val_loss=0.0688  acc=0.9986  val_acc=0.9799
Epoch 13/20  loss=0.0059  val_loss=0.0708  acc=0.9993  val_acc=0.9803
Epoch 14/20  loss=0.0053  val_los

Four numbers per epoch:

loss — cross-entropy on the 56,000 training samples. Fell from 0.16 → 0.002. The model got very confident on training data.
val_loss — cross-entropy on the 14,000 test samples. Fell from 0.18 → 0.079, then stopped improving around epoch 7 while train loss kept dropping.
acc — training accuracy. Reached 99.97% regarded as nearly perfect.
val_acc — test accuracy. Peaked around 98.2% and plateaued.

The gap between acc=0.9997 and val_acc=0.9814 is overfitting, the model has memorized training data slightly beyond what generalizes. It's mild but visible. The model learned the noise in the training set, not just the signal.

Been able to built something from scratch with no ML framework, just matrix multiplications, a ReLU, a softmax, and backprop by hand and it sits within 0.4% of what PyTorch gives with the same architecture. The remaining gap to 99%+ requires convolutional layers, which exploit the 2D spatial structure of images that the model throws away by flattening to 784 pixels.